# pctrl：最小化 HV / VH / DA / AD 四组双光子计数

这个示例复用 `qtools.pctrl.joint_optimize.CompositeMeasurement`：pctrl 负责 QWP-HWP-QWP 优化，composite measurement 负责切换 Alice/Bob basis 并读取四组 coincidence rate。

本 notebook 只需要配置参数和 loss function，不重复实现硬件测量循环。电机端口和 backend 从 `motor_config.yaml` 读取，basis 校准角度从 `polmeas_config.yaml` 读取。

## 1. 配置

请在实验目录中启动 Jupyter，并把 TDC 的窗口参数改成当前实验值。`TARGET` 是要优化的 pctrl channel。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qtools.motor import MotorConfigManager
from qtools.pctrl import PolarizationController
from qtools.pctrl.config import resolve_target
from qtools.pctrl.joint_optimize import CompositeMeasurement
from qtools.pctrl.optimize import optimize_channel
from qtools.pmeas.config import ConfigManager as PolmeasConfigManager
from qtools.pmeas.motor_manager import MotorManager

PCTRL_CONFIG = Path.cwd() / "pctrl_config.yaml"
MOTOR_CONFIG = Path.cwd() / "motor_config.yaml"
POLMEAS_CONFIG = Path.cwd() / "polmeas_config.yaml"
TARGET = "alice"

MEASUREMENT_KEYS = [("H", "V"), ("V", "H"), ("D", "A"), ("A", "D")]
MEASURE_DURATION = 1.0
MEASURE_SAMPLES = 3
CH_START, CH_STOP = 1, 4
DELAY = 100
WINDOW_START, WINDOW_STOP = 109, 110
THRESHVOLT = 0.6

MAXITER = 40
ANGLE_TOL = 1.0
STEP_SIZE_DEG = 15.0
SPEED_DEG_S = 10.0
SETTLE_TIME = 0.2

for path in (PCTRL_CONFIG, MOTOR_CONFIG, POLMEAS_CONFIG):
    print(f"{path}: {'OK' if path.exists() else 'missing'}")


## 2. 创建控制器和可复用的 composite measurement

`CompositeMeasurement` 已经实现 basis 电机切换、pairs 读取和评估历史。这里仅注入四组 basis 和自定义 loss：四组双光子计数的平均值。

In [ ]:
pctrl = PolarizationController(config_path=str(PCTRL_CONFIG))
motor_cfg = MotorConfigManager.resolve(MOTOR_CONFIG)
polmeas_cfg = PolmeasConfigManager.resolve(POLMEAS_CONFIG)
motor_mgr = MotorManager(motor_cfg)

# motor_config.yaml 提供实际连接参数；polmeas_config.yaml 提供 basis 角度。
# CompositeMeasurement 内部使用 optical path "alice" / "bob" 解析 basis。

def mean_four_basis_loss(counts):
    return float(np.mean([counts[key] for key in MEASUREMENT_KEYS]))

composite = CompositeMeasurement(
    meas_mgr=pctrl.meas_mgr,
    polmeas_cfg=polmeas_cfg,
    motor_mgr=motor_mgr,
    measurement_keys=MEASUREMENT_KEYS,
    loss_fn=mean_four_basis_loss,
)
print("controllers ready")


## 3. 运行优化

每次评估会自动完成 `HV → VH → DA → AD` 四次 pairs 测量。`minimize_metric=True` 使 Nelder-Mead 最小化四组平均计数。

In [ ]:
resolution = resolve_target(pctrl.cfg, TARGET)
if len(resolution.channels) != 1:
    raise ValueError(f"TARGET must resolve to one pctrl channel, got {TARGET!r}")
channel = resolution.channels[0]

def progress(iteration, angles, value, _raw):
    # composite.last_counts 是本次评估的四组原始 rate。
    counts = " ".join(
        f"{a}{b.lower()}={composite.last_counts[(a, b)]:.2f}"
        for a, b in MEASUREMENT_KEYS
    )
    print(f"eval {iteration:03d} | mean={value:.3f} cps | {counts}")

result = optimize_channel(
    cfg=pctrl.cfg,
    channel=channel,
    servo_mgr=pctrl.servo_mgr,
    meas_mgr=composite,
    minimize_metric=True,
    method="Nelder-Mead",
    maxiter=MAXITER,
    tol=ANGLE_TOL,
    step_size_deg=STEP_SIZE_DEG,
    speed_deg_s=SPEED_DEG_S,
    settle_time=SETTLE_TIME,
    duration=MEASURE_DURATION,
    samples=MEASURE_SAMPLES,
    measure_type="pairs",
    ch_start=CH_START,
    ch_stop=CH_STOP,
    delay=DELAY,
    window_start=WINDOW_START,
    window_stop=WINDOW_STOP,
    threshvolt=THRESHVOLT,
    callback=progress,
    verbose=True,
)
print(result.message)
print("optimal angles:", result.optimal_angles)


## 4. 查看四组计数和收敛过程

`history_rows` 由 `CompositeMeasurement` 记录，避免 notebook 自己维护第二套历史格式。

In [ ]:
rows = composite.history_rows
labels = ["".join(key) for key in MEASUREMENT_KEYS]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([row[0] for row in rows], [row[2] for row in rows], marker="o")
axes[0].set(xlabel="evaluation", ylabel="mean coincidence rate (cps)", title="loss")
axes[0].grid(True, alpha=0.3)

for index, label in enumerate(labels):
    key = MEASUREMENT_KEYS[index]
    axes[1].plot(
        [row[0] for row in rows],
        [row[1][key] for row in rows],
        marker="o",
        label=label,
    )
axes[1].set(xlabel="evaluation", ylabel="coincidence rate (cps)", title="HV / VH / DA / AD")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


## 5. 关闭连接

完成实验后执行。

In [ ]:
try:
    pctrl.close()
finally:
    motor_mgr.disconnect_all()
print("hardware connections closed")
